# Step 2 - preprocessing: T1w scans → HippUnfold → subfield volumes

Built from the team's original `BMEN600.ipynb` (DICOM → NIfTI, BIDS layout, renaming), extended to all 112
participants in `ADNI_Participants.xlsx` and all three groups. Helper code lives in `preprocess_hippunfold.py`;
all paths are set in `config.py` (change `DICOM_DIR`, `BIDS_DIR`, `HIPPUNFOLD_DIRS` if your data are on another drive).

| Cell | Does | Output |
|---|---|---|
| A | load the participant list | - |
| B | DICOM → NIfTI → BIDS (copies existing NIfTIs; never moves or deletes source files) | `BIDS_DIR/sub-XXX/anat/sub-XXX_T1w.nii.gz` |
| C | print the HippUnfold command - run it in a terminal | `HIPPUNFOLD_DIRS` |
| D | subfield volumes from HippUnfold's label maps | `data/derived/volumes.csv` |
| E | add intracranial volume (eTIV) | `eTIV` column in `volumes.csv` |
| F | QC sheet to fill in after looking at HippUnfold's QC images | `data/derived/qc.csv` |

Expected DICOM layout (as in the original notebook): `DICOM_DIR/<Group>/<sub-XXX>/.../anat/*.dcm`. Folders named by ADNI
subject ID (e.g. `002_S_1261`) also work. Re-running is safe: finished participants are skipped.

**Do not commit any outputs of this notebook** - they list participants. Clear outputs before committing.

In [ ]:
# A. Setup and participant list
import os, sys
from pathlib import Path
root = Path.cwd()
while not (root / "code" / "config.py").exists():
    if root.parent == root:
        raise FileNotFoundError("Open this notebook from inside the BMEN-600_code folder.")
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root / "code"))

import importlib, config as cfg, preprocess_hippunfold as pp
importlib.reload(cfg); importlib.reload(pp)
participants = pp.load_participants()
print(participants.Group.value_counts().to_dict(), "| DICOM_DIR:", cfg.DICOM_DIR, "| BIDS_DIR:", cfg.BIDS_DIR)

In [ ]:
# B. Convert to BIDS (needs: pip install dicom2nifti)
status = pp.convert_to_bids(participants)
print(status.status.str.split(":").str[0].value_counts().to_dict())
problems = status[status.status.str.startswith(("MISSING", "FAILED"))]
problems   # fix these before running HippUnfold

In [ ]:
# C. HippUnfold command - copy into a terminal (Docker shown; set docker=False for a local install).
# Optional: pp.hippunfold_command(["sub-001", "sub-002"]) to run a batch.
print(pp.hippunfold_command())

In [ ]:
# D. Subfield volumes from HippUnfold label maps (space-cropT1w, as HippUnfold recommends for volumetry)
vols, missing = pp.collect_volumes(participants)
print(f"{vols.filter(like='_L').notna().all(axis=1).sum()} of {len(participants)} participants have both hemispheres")
print("Missing:", missing[:10], "..." if len(missing) > 10 else "")

In [ ]:
# E. Intracranial volume. Put a CSV with Subject,eTIV at config.ETIV_FILE, or pass an ADNI UCSF FreeSurfer
# table (PTID, IMAGEUID, ST10CV columns), e.g. pp.add_etiv(participants, "data/raw/UCSFFSX7_<date>.csv")
if cfg.ETIV_FILE.exists():
    vols = pp.add_etiv(participants)
    print("eTIV missing for", vols.eTIV.isna().sum(), "participants")
else:
    print("No eTIV source yet:", cfg.ETIV_FILE)

In [ ]:
# F. QC sheet: open HippUnfold's qc/ images for each participant, enter QC_pass = 1 (good) or 0 (fail)
path, created = pp.make_qc_template(participants)
print(("Created " if created else "Already exists (not overwritten): ") + str(path))